# Bronze Layer Data Ingestion

Ingests raw CSV files from the `puc_data_specialist_de_2026_09.00-raw.brazil-ecommerce` volume into Delta Lake tables in the `01-bronze` schema. Each source file is converted to a typed Delta table with an `_ingested_at` metadata column for auditability.

Many of the code used in this Notebook has been provided by Databricks Labs training available here: https://github.com/IvanJPC/databricks-labs-ago-2026-data-ingestion/

The focus is use SQL statements instead of Apache Spark. Therefore, this notebook uses Apache Spark when executing SQL statements. It enables the power of using Python logical code with SQL. 

A improvement for this work is to change some of the SQL statements to use Spark only like Auto Loader or SQL Copy Into statement

In [0]:
%sql
--Setting UP Notebook variables and useful SQL commands to reduce typos
USE CATALOG puc_data_specialist_de_2026_09;
USE SCHEMA `01-bronze`;

In [0]:
%sql
-- Checking all file metadata columns to be used later
SELECT _metadata.*
FROM read_files('/Volumes/puc_data_specialist_de_2026_09/00-raw/brazil-ecommerce/2026-09-22/olist_customers_dataset.csv', format => 'csv')

In [0]:
from datetime import datetime

dt = datetime.now().strftime("%Y-%m-%d")

base_path = f"/Volumes/puc_data_specialist_de_2026_09/00-raw/brazil-ecommerce/{dt}"
print(base_path)

Understanding each file schema and data volume to create the SQL create table commands for each CSV 

In [0]:

csv_files = [
     "olist_customers_dataset.csv"
    , "olist_orders_dataset.csv"
    , "olist_geolocation_dataset.csv"
    , "olist_order_items_dataset.csv"
    , "olist_order_payments_dataset.csv"
    , "olist_order_reviews_dataset.csv"
    , "olist_products_dataset.csv"
    , "olist_sellers_dataset.csv"
    , "product_category_name_translation.csv"
]

for file_name in csv_files:
    df = spark.read.csv(f"{base_path}/{file_name}", header=True, inferSchema=True)
    print(f"=== {file_name} ===")
    print(f"Columns: {df.columns}")
    print(f"Row count: {df.count()}")
    df.show(5)
    print("---------------------------------------------")

    

Create all tables dynamicaly using SQL Statement Create Table AS (CTAS) and Python/Spark 

Some considerations:
- The table name for Bronze will be the name of the file without the olist prefix and dataset suffix
- When creating a table from a file using CTAS, the table schema is auto generated, unless a schema is specified.
- A few columns has been wrongly created as int instead of string by CTAS. These columns has been specified to change to string on some cases, like BR zip code
- Each table schema has been reviewed and adjusted when needed
- New datetime column has been added during this process to record when data was ingested (ingestion_time)
- Metadata columns - Also important to track the source file and the file modification time (file_modification_time, source_file)
- It has been oberserved that timestamp values have been ingested using timestamp type instead of string through the read_file 
    - The CSV files do not contain any timezone information. Assuming BRTMZ for all timestamp fields
    - This is causing a wrong timezone conversion error as Databricks understands the value is on GMT +0 instead of GMT -3 (BRTMZ)
    - All timestamp values will be forced to be ingested into Bronze as string. The converstion to timestamp will be handled throughout Bronze to Silver refinement
- Columns metatada will be added throughout Bronze to Silver refinement

Checking Rescue Data and managing data schema mismatch.

The rescued data column contains any data that isn’t parsed for the following reasons:

- The column is missing from the schema.
- Type mismatches
- Case mismatches

More information about rescue Data column is available here: https://docs.databricks.com/aws/en/ingestion/cloud-object-storage/auto-loader/schema#rescue

In [0]:
total_malformed_result = 0

for file_name in csv_files:

    bronze_table_name = file_name.removeprefix("olist_").removesuffix(".csv").removesuffix("_dataset")
    print(f"Bronze table name to be created: {bronze_table_name}")
    
    query = f"""DROP TABLE IF EXISTS {bronze_table_name};"""
    spark.sql(query)

    
    query = f""" CREATE TABLE {bronze_table_name} AS
      SELECT *,
      _metadata.file_modification_time AS file_modification_time,
      _metadata.file_path AS source_file, 
      current_timestamp() as ingestion_time  
      FROM read_files(
             '{base_path}/{file_name}',
             format => 'csv',
             schemaHints => 'customer_zip_code_prefix string, order_purchase_timestamp string, order_approved_at string
             , order_delivered_carrier_date string, order_delivered_customer_date string, order_estimated_delivery_date string
             , geolocation_zip_code_prefix string, geolocation_lat string, geolocation_lng string
             , shipping_limit_date string
             , review_creation_date string, review_answer_timestamp string
             , product_weight_g double, product_length_cm double, product_height_cm double, product_width_cm double
             , seller_zip_code_prefix string'
         );"""
    spark.sql(query)

    table_description = f"Bronze layer table first ingestion from source file(s): {base_path}/{file_name}"

    # Adding Table description metadata
    description_escaped = table_description.replace("'", "''")
    query = f"""COMMENT ON TABLE {bronze_table_name} IS '{description_escaped}';"""
    spark.sql(query)

    # Describe statement used to review the generated schema from each table -> ingested file and fix schema issues when needed
    # Disable to reduce the result verbose
    # query = f"""DESCRIBE TABLE EXTENDED {bronze_table_name};"""
    # display(spark.sql(query))

    query = f"""SELECT count(_rescued_data) AS malformed_data_count FROM {bronze_table_name} where _rescued_data is not null;"""
    malformed_result = spark.sql(query).collect()[0]["malformed_data_count"]
    total_malformed_result = total_malformed_result + malformed_result
    if malformed_result != 0:
        ## instead of reasing an exception, continue the process and report the error
        print(f"Validation failed for {bronze_table_name}: found {malformed_result} malformed rows, expected 0.")
    else:
        print(f"Validation passed for {bronze_table_name}: 0 malformed rows found.")
    
if (total_malformed_result > 0 ):
    raise Exception(f"Validation failed: found {total_malformed_result} malformed rows, expected 0.")
else:
    print(f"Final validation passed: 0 malformed rows found.")

In [0]:
# Add table descriptions based on source_file column values
for file_name in csv_files:
    bronze_table_name = file_name.removeprefix("olist_").removesuffix(".csv").removesuffix("_dataset")

    # Retrieve the source_file value(s) for this table
    source_files_df = spark.sql(f"SELECT DISTINCT source_file FROM {bronze_table_name}")
    source_files = [row["source_file"] for row in source_files_df.collect()]
    source_files_str = ", ".join(source_files)

    description = f"Bronze layer table ingested from source file(s): {source_files_str}"

    # Escape single quotes in the description for SQL
    description_escaped = description.replace("'", "''")

    spark.sql(f"COMMENT ON TABLE {bronze_table_name} IS '{description_escaped}'")
    print(f"Table {bronze_table_name}: description set to '{description}'")